# DynamoDB Hands-On Lab

Create a table, store documents with different attributes, query with both the API and PartiQL, load into DataFrames, and clean up.

## Setup — Get Your AWS Credentials

You need two values from your local machine to paste into **Colab Secrets**.

### Find your credentials
Open a terminal and run:
```
cat ~/.aws/credentials
```
You should see:
```
[default]
aws_access_key_id = AKIA...
aws_secret_access_key = Y+Co...
```

### Add secrets to Colab
1. Click the **🔑 key icon** in the left sidebar
2. Click **+ Add new secret** and create these two:
   - `AWS_ACCESS_KEY_ID` — starts with `AKIA...` (**required**)
   - `AWS_SECRET_ACCESS_KEY` — long string of letters/numbers (**required**)
3. Toggle **Notebook access** ON for each secret
4. Run the cell below — it should print `Connected as: arn:aws:sts::...`

In [6]:
!pip install -q boto3

import boto3, os
from decimal import Decimal
from botocore.exceptions import ClientError

try:
    from google.colab import userdata
    os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
    os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
    print('Loaded credentials from Colab Secrets')
except ImportError:
    print('Not in Colab - using default AWS credential chain')

sts = boto3.client('sts', region_name='us-east-1')
identity = sts.get_caller_identity()
print(f"Connected as: {identity['Arn']}")
print(f"Account: {identity['Account']}")

REGION = 'us-east-1'
TABLE_NAME = 'student-directory-lab'
dynamodb = boto3.resource('dynamodb', region_name=REGION)
print(f'DynamoDB ready (region: {REGION})')

Loaded credentials from Colab Secrets
Connected as: arn:aws:iam::590184128502:user/leonardo-admin
Account: 590184128502
DynamoDB ready (region: us-east-1)


## Step 1: Create a Table
We define only the partition key (`student_id`). No columns for name, gpa, etc.
Different items can have completely different attributes - that is the document model.

In [7]:
try:
    table = dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[{'AttributeName': 'student_id', 'KeyType': 'HASH'}],
        AttributeDefinitions=[{'AttributeName': 'student_id', 'AttributeType': 'S'}],
        BillingMode='PAY_PER_REQUEST'
    )
    print(f'Creating table "{TABLE_NAME}"...')
    table.wait_until_exists()
    print(f'Table created! Status: {table.table_status}')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceInUseException':
        print(f'Table "{TABLE_NAME}" already exists.')
        table = dynamodb.Table(TABLE_NAME)
    else: raise
print(f'ARN: {table.table_arn}')

Creating table "student-directory-lab"...
Table created! Status: CREATING
ARN: arn:aws:dynamodb:us-east-1:590184128502:table/student-directory-lab


## Step 2: Put Documents
Each item is a JSON document. Notice each student has **different attributes**:
- Alice has `contact` (nested map)
- Bob has `thesis_title`
- Carol has `scholarship`
- David has `minor`

No ALTER TABLE needed. Just include whatever attributes are relevant.

In [8]:
students = [
    {'student_id': 'stu-001', 'name': 'Alice Johnson', 'major': 'Data Science',
     'year': 3, 'gpa': Decimal('3.85'), 'skills': ['python', 'sql', 'statistics'],
     'contact': {'email': 'alice@example.edu', 'campus': 'main'}},
    {'student_id': 'stu-002', 'name': 'Bob Martinez', 'major': 'Computer Science',
     'year': 4, 'gpa': Decimal('3.72'), 'skills': ['java', 'python', 'aws'],
     'thesis_title': 'Distributed Cache Invalidation'},
    {'student_id': 'stu-003', 'name': 'Carol Chen', 'major': 'Data Science',
     'year': 2, 'gpa': Decimal('3.95'), 'skills': ['r', 'python', 'tableau'],
     'scholarship': True},
    {'student_id': 'stu-004', 'name': 'David Kim', 'major': 'Applied Mathematics',
     'year': 3, 'gpa': Decimal('3.60'), 'skills': ['python', 'matlab'],
     'minor': 'Data Science'}
]

for s in students:
    table.put_item(Item=s)
    common = {'student_id', 'name', 'major', 'year', 'gpa', 'skills'}
    unique = set(s.keys()) - common
    extra = f'  (unique: {unique})' if unique else ''
    print(f'Created: {s["name"]}{extra}')

print(f'\nStored {len(students)} documents with different attribute shapes')

Created: Alice Johnson  (unique: {'contact'})
Created: Bob Martinez  (unique: {'thesis_title'})
Created: Carol Chen  (unique: {'scholarship'})
Created: David Kim  (unique: {'minor'})

Stored 4 documents with different attribute shapes


## Step 3: Get an Item
Fast lookup by partition key. SQL equivalent: `SELECT * FROM students WHERE student_id = 'stu-001'`

In [12]:
response = table.get_item(Key={'student_id': 'stu-002'})
if 'Item' in response:
    item = response['Item']
    print(f'Name:    {item["name"]}')
    print(f'Major:   {item["major"]}')
    print(f'GPA:     {float(item["gpa"])}')
    print(f'Skills:  {item["skills"]}')
    print(f'Contact: {item.get("contact", "N/A")}')

# Non-existent item returns empty, not an error
resp2 = table.get_item(Key={'student_id': 'stu-999'})
print(f'\nstu-999 found: {"Item" in resp2}')

Name:    Bob Martinez
Major:   Computer Science
GPA:     3.72
Skills:  ['java', 'python', 'aws']
Contact: N/A

stu-999 found: False


## Step 4: Update Attributes
Unlike `put_item` (replaces entire item), `update_item` changes specific attributes.
The expression syntax is verbose but prevents injection and handles reserved words.

In [13]:
# Update year and GPA
print('Updating Alice: year 3->4, GPA 3.85->3.90')
table.update_item(
    Key={'student_id': 'stu-001'},
    UpdateExpression='SET #yr = :y, gpa = :g',
    ExpressionAttributeNames={'#yr': 'year'},
    ExpressionAttributeValues={':y': 4, ':g': Decimal('3.90')}
)
item = table.get_item(Key={'student_id': 'stu-001'})['Item']
print(f'After: year={item["year"]}, gpa={float(item["gpa"])}')

# Add a NEW attribute (no ALTER TABLE needed!)
print('\nAdding honors attribute...')
table.update_item(
    Key={'student_id': 'stu-001'},
    UpdateExpression='SET honors = :h',
    ExpressionAttributeValues={':h': True}
)

# Append to a list
print('Appending "aws" to skills...')
table.update_item(
    Key={'student_id': 'stu-001'},
    UpdateExpression='SET skills = list_append(skills, :new)',
    ExpressionAttributeValues={':new': ['aws']}
)
item = table.get_item(Key={'student_id': 'stu-001'})['Item']
print(f'Skills: {item["skills"]}')

Updating Alice: year 3->4, GPA 3.85->3.90
After: year=4, gpa=3.9

Adding honors attribute...
Appending "aws" to skills...
Skills: ['python', 'sql', 'statistics', 'aws']


## Step 5: Scan
Scan reads **every** item, then filters. Fine for small tables, expensive for millions of rows.

In [14]:
from boto3.dynamodb.conditions import Attr

print('All items:')
for item in table.scan()['Items']:
    print(f'  {item["student_id"]}: {item["name"]}  attrs={list(item.keys())}')

print('\nData Science students:')
resp = table.scan(FilterExpression=Attr('major').eq('Data Science'))
for item in resp['Items']:
    print(f'  {item["name"]} (GPA: {float(item["gpa"])})')

print('\nGPA > 3.80:')
resp = table.scan(FilterExpression=Attr('gpa').gt(Decimal('3.80')))
for item in resp['Items']:
    print(f'  {item["name"]} (GPA: {float(item["gpa"])})')

All items:
  stu-001: Alice Johnson  attrs=['honors', 'gpa', 'contact', 'year', 'major', 'skills', 'student_id', 'name']
  stu-003: Carol Chen  attrs=['gpa', 'scholarship', 'year', 'major', 'skills', 'student_id', 'name']
  stu-004: David Kim  attrs=['minor', 'gpa', 'year', 'major', 'skills', 'student_id', 'name']
  stu-002: Bob Martinez  attrs=['gpa', 'year', 'major', 'skills', 'student_id', 'thesis_title', 'name']

Data Science students:
  Alice Johnson (GPA: 3.9)
  Carol Chen (GPA: 3.95)

GPA > 3.80:
  Alice Johnson (GPA: 3.9)
  Carol Chen (GPA: 3.95)


## Step 6: PartiQL — SQL Syntax for DynamoDB

DynamoDB also supports **PartiQL**, a SQL-compatible query language. This lets you write familiar `SELECT`, `INSERT`, and `UPDATE` statements instead of using the API.

**Key difference:** PartiQL uses the **low-level client** (`boto3.client`), not the resource (`boto3.resource`). The responses use DynamoDB's type descriptors (`{'S': 'value'}` instead of just `'value'`).

In [15]:
# PartiQL uses the low-level CLIENT, not the resource
client = boto3.client('dynamodb', region_name=REGION)

# SELECT one item by partition key
print('=== PartiQL: Get one student ===')
response = client.execute_statement(
    Statement=f'SELECT * FROM "{TABLE_NAME}" WHERE student_id=?',
    Parameters=[{'S': 'stu-001'}]
)
for item in response['Items']:
    print(f"  {item['name']['S']} — {item['major']['S']} — GPA: {item['gpa']['N']}")

# SELECT with a filter (works like a WHERE clause)
print('\n=== PartiQL: Data Science students ===')
response = client.execute_statement(
    Statement=f'SELECT name, gpa FROM "{TABLE_NAME}" WHERE major=?',
    Parameters=[{'S': 'Data Science'}]
)
for item in response['Items']:
    print(f"  {item['name']['S']} — GPA: {item['gpa']['N']}")

# UPDATE with PartiQL
print('\n=== PartiQL: Update Bob\'s year ===')
client.execute_statement(
    Statement=f'UPDATE "{TABLE_NAME}" SET year=? WHERE student_id=?',
    Parameters=[{'N': '4'}, {'S': 'stu-002'}]
)
# Verify the update
resp = client.execute_statement(
    Statement=f'SELECT name, year FROM "{TABLE_NAME}" WHERE student_id=?',
    Parameters=[{'S': 'stu-002'}]
)
for item in resp['Items']:
    print(f"  {item['name']['S']} is now year {item['year']['N']}")

=== PartiQL: Get one student ===
  Alice Johnson — Data Science — GPA: 3.9

=== PartiQL: Data Science students ===
  Alice Johnson — GPA: 3.9
  Carol Chen — GPA: 3.95

=== PartiQL: Update Bob's year ===
  Bob Martinez is now year 4


## Step 7: DynamoDB → Pandas DataFrame

As data scientists, you want your data in DataFrames. Here are two methods:
- **Manual:** Scan → convert Decimals → `pd.DataFrame`
- **awswrangler:** PartiQL query → DataFrame in one line

In [16]:
import pandas as pd

# ---- Method 1: Manual scan → DataFrame ----
response = table.scan()

def decimals_to_floats(item):
    """DynamoDB returns Decimal; Pandas wants float."""
    return {k: float(v) if isinstance(v, Decimal) else v
            for k, v in item.items()}

items = [decimals_to_floats(i) for i in response['Items']]
df = pd.DataFrame(items)

print('=== Manual Scan → DataFrame ===')
print(df[['student_id', 'name', 'major', 'gpa']].to_string(index=False))
print(f'\nMean GPA: {df["gpa"].mean():.2f}')
print(f'\nGPA by major:')
print(df.groupby('major')['gpa'].mean())

=== Manual Scan → DataFrame ===
student_id          name               major  gpa
   stu-001 Alice Johnson        Data Science 3.90
   stu-003    Carol Chen        Data Science 3.95
   stu-004     David Kim Applied Mathematics 3.60
   stu-002  Bob Martinez    Computer Science 3.72

Mean GPA: 3.79

GPA by major:
major
Applied Mathematics    3.600
Computer Science       3.720
Data Science           3.925
Name: gpa, dtype: float64


In [ ]:
# ---- Method 2: awswrangler (AWS SDK for pandas) ----
!pip install -q awswrangler

import awswrangler as wr

# PartiQL query → DataFrame in ONE line
print('=== awswrangler: Full table ===')
df_wr = wr.dynamodb.read_partiql_query(
    query=f'SELECT * FROM "{TABLE_NAME}"',
    boto3_session=boto3.Session(region_name=REGION)
)
print(df_wr)

print('\n=== awswrangler: Filtered query ===')
ds_df = wr.dynamodb.read_partiql_query(
    query=f'SELECT name, gpa FROM "{TABLE_NAME}" WHERE major=?',
    parameters=['Data Science'],
    boto3_session=boto3.Session(region_name=REGION)
)
print(ds_df)

## Step 8: Delete

In [17]:
print('Deleting David (stu-004)...')
table.delete_item(Key={'student_id': 'stu-004'})
resp = table.get_item(Key={'student_id': 'stu-004'})
print(f'Confirmed deleted: {"Item" not in resp}')

print(f'\nRemaining: {len(table.scan()["Items"])} items')

Deleting David (stu-004)...
Confirmed deleted: True

Remaining: 3 items


## Step 9: Mini Challenge
1. Add 2+ new student records with **unique attributes** (e.g., `club`, `graduation_date`, `internship` map)
2. Use **PartiQL** to query for students matching criteria you choose
3. Load all items into a DataFrame and compute a **summary statistic** (mean, count, etc.)
4. **Screenshot your output** for Canvas submission

In [28]:
# YOUR CODE: Add new students with unique attributes

table.put_item(Item={'student_id':'stu-005', 'name':'Ryan Carpone', 'skills':['soccer','python','Java'], 'internship':'yes'})
table.put_item(Item={'student_id':'stu-006', 'name':'Jhonny Maccaroni', 'skills':'Python', 'gpa':'3.8' ,'internship':'yes'})




{'ResponseMetadata': {'RequestId': 'LF9PUSSTG32Q0U1EP1GP6H8E77VV4KQNSO5AEMVJF66Q9ASUAAJG',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'server': 'Server',
   'date': 'Tue, 17 Feb 2026 02:24:52 GMT',
   'content-type': 'application/x-amz-json-1.0',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'LF9PUSSTG32Q0U1EP1GP6H8E77VV4KQNSO5AEMVJF66Q9ASUAAJG',
   'x-amz-crc32': '2745614147'},
  'RetryAttempts': 0}}

In [32]:
# YOUR CODE: PartiQL query
print('=== PartiQL: Get one student ===')
response = client.execute_statement(
    Statement=f'SELECT * FROM "{TABLE_NAME}" WHERE student_id=?',
    Parameters=[{'S': 'stu-005'}]
)
for item in response['Items']:
    name = item['name']['S']
    skills = [x['S'] for x in item['skills']['L']]
    internship = item.get('internship', {}).get('S', None)
    gpa = item.get('gpa', {}).get('N', None)

    print(f"  {name} — {skills} — internship: {internship} — GPA: {gpa}")




=== PartiQL: Get one student ===
  Ryan Carpone — ['soccer', 'python', 'Java'] — internship: yes — GPA: None


In [34]:
# YOUR CODE: Load into DataFrame and compute a stat
items_all = []
response = table.scan()
items_all.extend(response.get("Items", []))

while "LastEvaluatedKey" in response:
    response = table.scan(ExclusiveStartKey=response["LastEvaluatedKey"])
    items_all.extend(response.get("Items", []))

def decimals_to_floats(item):
    """DynamoDB returns Decimal; Pandas wants float."""
    return {k: float(v) if isinstance(v, Decimal) else v
            for k, v in item.items()}

items = [decimals_to_floats(i) for i in items_all]
df = pd.DataFrame(items)
mask_known = df["internship"].notna()
rate_yes_known = (df.loc[mask_known, "internship"] == "yes").mean() if mask_known.any() else float("nan")

rate_yes_total = (df["internship"] == "yes").sum() / len(df) if len(df) else float("nan")


cols_to_show = [c for c in ["student_id", "name", "internship", "major", "gpa"] if c in df.columns]
print(df[cols_to_show].to_string(index=False))

yes_count = (df["internship"] == "yes").sum()
known_count = mask_known.sum()
total_count = len(df)

print(f"\nYes count: {yes_count}")
print(f"Known internship rows: {known_count} / {total_count}")
print(f"\nInternship==yes rate (among known): {rate_yes_known:.3f} ({rate_yes_known*100:.1f}%)")
print(f"Internship==yes rate (over total):  {rate_yes_total:.3f} ({rate_yes_total*100:.1f}%)")

student_id             name internship            major   gpa
   stu-001    Alice Johnson        NaN     Data Science   3.9
   stu-003       Carol Chen        NaN     Data Science  3.95
   stu-006 Jhonny Maccaroni        yes              NaN   3.8
   stu-005     Ryan Carpone        yes              NaN   NaN
   stu-002     Bob Martinez        NaN Computer Science  3.72

Yes count: 2
Known internship rows: 2 / 5

Internship==yes rate (among known): 1.000 (100.0%)
Internship==yes rate (over total):  0.400 (40.0%)


## Cleanup
**Run this when done** to delete the table and avoid charges.

In [ ]:
print(f'Deleting table "{TABLE_NAME}"...')
table.delete()
table.wait_until_not_exists()
print('Table deleted. Verify in AWS Console that no tables remain.')